Source: https://github.com/gregrahn/join-order-benchmark

In [32]:
import duckdb
import requests
import tarfile
import os
import io
import shutil
import re

In [ ]:
DATA_URL = "http://event.cwi.nl/da/job/imdb.tgz"
TAR_FILE_NAME = "imdb.tgz"

EXTRACT_DIR = "imdb_data"
EXPORT_DIR = "duckdb_export"
DUCKDB_FILE = "job_benchmark.duckdb"

TABLES_TO_INCLUDE = ['movie_companies',
                     'person_info']

In [ ]:
TABLE_NAMES = [
    "aka_name", "aka_title", "cast_info", "char_name", "comp_cast_type",
    "company_name", "company_type", "complete_cast", "info_type", "keyword",
    "kind_type", "link_type", "movie_companies", "movie_info",
    "movie_info_idx", "movie_keyword", "movie_link", "name", "person_info",
    "role_type", "title"
]

In [ ]:
def download_and_extract_data():
    """Baixa e extrai os dados .tgz se ainda não existirem."""
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    
    first_csv = os.path.join(EXTRACT_DIR, f"{TABLE_NAMES[0]}.csv")
    if os.path.exists(first_csv):
        print(f"Diretório '{EXTRACT_DIR}' já contém dados. Pulando download e extração.")
        return

    print(f"Baixando dados de {DATA_URL}...")
    try:
        response = requests.get(DATA_URL, stream=True)
        response.raise_for_status()
        
        with io.BytesIO(response.content) as tar_buffer:
            with tarfile.open(fileobj=tar_buffer, mode="r:gz") as tar:
                print(f"Extraindo arquivos para '{EXTRACT_DIR}'...")
                tar.extractall(path=EXTRACT_DIR)
        print("Download e extração concluídos.")
        
    except requests.exceptions.RequestException as e:
        print(f"Erro ao baixar os dados: {e}")
        exit()
    except tarfile.TarError as e:
        print(f"Erro ao extrair o arquivo tar: {e}")
        exit()

In [ ]:
def parse_schema_file(filepath, table_list):
    schema = {}
    current_table = None
    
    table_regex = re.compile(r"CREATE TABLE (\w+) \(", re.IGNORECASE)
    
    print(f"Analisando schema de {filepath}...")
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                
                table_match = table_regex.match(line)
                if table_match:
                    current_table = table_match.group(1).lower()
                    
                    if current_table in table_list:
                        schema[current_table] = {}
                    else:
                        current_table = None
                    continue
                
                if current_table:
                    if line.startswith(');'):
                        current_table = None
                        continue
                        
                    parts = line.split(maxsplit=2)
                    if len(parts) >= 2 and not line.upper().startswith('PRIMARY KEY') and not line.upper().startswith('CONSTRAINT'):
                        col_name = parts[0].strip('`"')
                        
                        col_type = parts[1].split()[0].strip().replace(',', '')
                        
                        col_type_lower = col_type.lower()
                        if 'integer' in col_type_lower:
                            db_type = 'INTEGER'
                        elif 'char' in col_type_lower:
                            db_type = 'VARCHAR'
                        elif 'text' in col_type_lower:
                            db_type = 'VARCHAR'
                        elif 'date' in col_type_lower:
                            db_type = 'DATE'
                        elif 'decimal' in col_type_lower:
                            db_type = 'DECIMAL'
                        else:
                            db_type = col_type.upper() 
                        
                        schema[current_table][col_name] = db_type

    except FileNotFoundError:
        print(f"ERRO: Arquivo de schema {filepath} não encontrado.")
        return None
    except Exception as e:
        print(f"Erro ao analisar o schema: {e}")
        return None

    print(f"Análise do schema concluída. {len(schema)} tabelas mapeadas.")
    return schema

In [ ]:
def load_tables_to_duckdb(con):
    schema_file_path = os.path.join(EXTRACT_DIR, 'schematext.sql')
    
    schema_definitions = parse_schema_file(schema_file_path, TABLE_NAMES)
    
    if schema_definitions is None:
        print("ERRO: Não foi possível analisar o arquivo de schema. Abortando carregamento.")
        return
    
    print(f"\nCarregando tabelas para o banco de dados DuckDB '{DUCKDB_FILE}'...")
    
    for table_name in TABLE_NAMES:
        if table_name not in TABLES_TO_INCLUDE:
            print(f"  Pulando tabela '{table_name}' conforme configuração.")
            continue
      
        csv_path = os.path.join(EXTRACT_DIR, f"{table_name}.csv")
        
        if not os.path.exists(csv_path):
            print(f"  AVISO: Arquivo {csv_path} não encontrado. Tabela '{table_name}' será pulada.")
            continue

        if table_name not in schema_definitions or not schema_definitions[table_name]:
            print(f"  AVISO: Schema para tabela '{table_name}' não encontrado no arquivo sql. Pulando.")
            continue
            
        columns = schema_definitions[table_name]
        columns_arg = str(columns)

        print(f"  Carregando tabela '{table_name}' de {csv_path}...")
        
        try:
            con.sql(f"""
                CREATE OR REPLACE TABLE {table_name} AS 
                SELECT * FROM read_csv_auto(
                    '{csv_path}',
                    HEADER=False,
                    DELIM=',',
                    QUOTE='\"',
                    IGNORE_ERRORS=True,
                    COLUMNS={columns_arg}
                )
            """)
        except duckdb.Error as e:
            print(f"    Erro ao carregar a tabela {table_name}: {e}")
            
    print("Carregamento das tabelas no DuckDB concluído.")

In [ ]:
def export_tables_to_csv(con):
    """Exporta todas as tabelas do DuckDB para arquivos CSV."""
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print(f"\nExportando tabelas do DuckDB para o diretório '{EXPORT_DIR}'...")

    for table_name in TABLE_NAMES:
        export_path = os.path.join(EXPORT_DIR, f"{table_name}.csv")
        print(f"  Exportando tabela '{table_name}' para {export_path}...")
        
        try:
            con.sql(f"""
                COPY {table_name} TO '{export_path}' (HEADER, DELIMITER ',')
            """)
        except duckdb.Error as e:
            try:
                con.sql(f"DESCRIBE {table_name};")
                print(f"    Erro ao exportar a tabela {table_name}: {e}")
            except duckdb.Error:
                print(f"    AVISO: Tabela '{table_name}' não existe no DB, não pode exportar.")

    print("Exportação concluída.")

In [ ]:
download_and_extract_data()

con = duckdb.connect(database=DUCKDB_FILE, read_only=False)

load_tables_to_duckdb(con)

export_tables_to_csv(con)

con.close()
os.remove(DUCKDB_FILE)

schema_source_path = os.path.join(EXTRACT_DIR, 'schematext.sql')
schema_dest_path = os.path.join(EXPORT_DIR, 'schematext.sql')
if os.path.exists(schema_source_path):
  print(f"\nCopiando {schema_source_path} para {schema_dest_path}...")
  shutil.copy(schema_source_path, schema_dest_path)
else:
  print(f"\nAVISO: Arquivo de schema {schema_source_path} não encontrado. Pulando cópia.")

if os.path.exists(EXTRACT_DIR):
  print(f"Removendo diretório de extração temporário: {EXTRACT_DIR}...")
  shutil.rmtree(EXTRACT_DIR)

print(f"\nProcesso finalizado com sucesso!")
print(f"Banco de dados DuckDB salvo em: {DUCKDB_FILE}")
print(f"CSVs exportados salvos em: {EXPORT_DIR}")

In [ ]:
for filename in os.listdir(EXPORT_DIR):
  if filename.endswith(".csv"):
    old_path = os.path.join(EXPORT_DIR, filename)
    new_filename = f"imdb_{filename}"
    new_path = os.path.join(EXPORT_DIR, new_filename)
    
    print(f"  Renomeando '{filename}' para '{new_filename}'")
    os.rename(old_path, new_path)

print("Renomeação concluída.")

In [ ]:
DATA_DIR = '../data'

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Movendo arquivos de '{EXPORT_DIR}' para '{DATA_DIR}'...")

for filename in os.listdir(EXPORT_DIR):
  source_path = os.path.join(EXPORT_DIR, filename)
  if os.path.isfile(source_path):
    destination_path = os.path.join(DATA_DIR, filename)
    print(f"  Movendo '{filename}'...")
    shutil.move(source_path, destination_path)

print("Movimentação de arquivos concluída.")

if os.path.exists(EXPORT_DIR):
  print(f"\nRemovendo o diretório '{EXPORT_DIR}'...")
  shutil.rmtree(EXPORT_DIR)
  print("Diretório removido.")